# Step 1 — Fix the bounding boxes

Only one goal in this notebook: **get one page's bounding boxes correct.** Nothing about text extraction, nothing about Gemini, nothing about ground truth data entry. Just boxes.

## Why this comes first

The current OpenCV detector produces boxes that overlap, miss entries, or include multiple entries per box. Any text extraction on top of bad boxes will inherit the mess. Fix the foundation, then build on it.

## Three modes in the tool

1. **Auto** — the heuristic draws its best-guess boxes
2. **Manual edit** — drag corners to resize, click to delete, drag on empty space to draw a new box
3. **Save** — download the corrected boxes as JSON

The saved JSON is the ground truth for one page's box layout. It becomes the input for step 2 (typing field values while looking at those boxes).

## How to run this notebook

- Section 0 — setup (once)
- Section 1 — run the auto-detector on your page (produces initial boxes)
- Section 2 — open the box editor HTML in a browser, fix the boxes by hand
- Section 3 — verify the saved boxes look right

## 0. Setup

In [1]:
from pathlib import Path
import json
import cv2
import numpy as np

# Point this to your page image
BASE = Path(r"C:\Users\ABHIRAMI.K\Documents\RICE\Summer 2026\Fondren Internship\Data\image")
SAMPLE_PAGE = BASE / "1900-1901 (page 200).png"

# Output folder
OUT = Path("output_box_fixing")
OUT.mkdir(exist_ok=True)

assert SAMPLE_PAGE.exists(), f"Page not found: {SAMPLE_PAGE}"
print("Setup OK. Page:", SAMPLE_PAGE.name)

Setup OK. Page: 1900-1901 (page 200).png


## 1. Auto-detect initial boxes

Run the existing OpenCV heuristic to get a starting set of boxes. These will be wrong in many places — that's expected. The next step is manual correction.

In [2]:
def preprocess(image_path):
    img = cv2.imread(str(image_path))
    return cv2.fastNlMeansDenoisingColored(img, None, 7, 7, 7, 21)


def detect_columns(img):
    """Return (left_column, right_column) bounding boxes."""
    h, w = img.shape[:2]
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, binary = cv2.threshold(gray, 0, 255,
                              cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    side_margin = int(w * 0.08)
    top_margin = int(h * 0.08)
    bottom_margin = int(h * 0.92)
    core = binary[top_margin:bottom_margin, side_margin:w - side_margin]
    vproj = core.sum(axis=0)
    window = max(5, core.shape[1] // 60)
    smoothed = np.convolve(vproj, np.ones(window)/window, mode="same")
    mid_start = len(smoothed) // 3
    mid_end = 2 * len(smoothed) // 3
    gap_x = side_margin + mid_start + int(np.argmin(smoothed[mid_start:mid_end]))
    buffer = int(w * 0.01)
    return ((side_margin, top_margin, gap_x - buffer, bottom_margin),
            (gap_x + buffer, top_margin, w - side_margin, bottom_margin))


def detect_entries_in_column(col_img):
    """Return list of (y_start, y_end) for entries within one column."""
    gray = cv2.cvtColor(col_img, cv2.COLOR_BGR2GRAY)
    _, binary = cv2.threshold(gray, 0, 255,
                              cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    h, w = binary.shape
    leftmost = np.full(h, w, dtype=np.int32)
    for y in range(h):
        nz = np.where(binary[y, :] > 0)[0]
        if len(nz):
            leftmost[y] = nz[0]
    text_rows = leftmost[leftmost < w]
    if len(text_rows) < 10:
        return []
    left_margin = int(np.median(text_rows))
    tol = max(6, int(w * 0.015))
    at_margin = (leftmost <= left_margin + tol) & (leftmost < w)
    indented = (leftmost > left_margin + tol) & (leftmost < w)
    empty = (leftmost == w)
    starts = [y for y in range(h)
              if at_margin[y] and (y == 0 or empty[y-1] or indented[y-1])]
    entries = []
    for i, s in enumerate(starts):
        e = starts[i+1] if i+1 < len(starts) else h
        while e > s and empty[e-1]:
            e -= 1
        if e - s >= max(10, int(h * 0.004)):
            entries.append((s, e))
    return entries


def auto_detect_boxes(image_path):
    """Return list of {id, x1, y1, x2, y2} dicts in full-page coordinates."""
    img = preprocess(image_path)
    left_col, right_col = detect_columns(img)
    boxes = []
    box_id = 0
    for col_label, (cx1, cy1, cx2, cy2) in [("L", left_col), ("R", right_col)]:
        col_img = img[cy1:cy2, cx1:cx2]
        for (y1, y2) in detect_entries_in_column(col_img):
            # Small padding so text is never clipped
            pad_top, pad_bot, pad_side = 8, 8, 4
            boxes.append({
                "id":    box_id,
                "x1":    max(0, cx1 - pad_side),
                "y1":    max(0, cy1 + y1 - pad_top),
                "x2":    min(img.shape[1], cx2 + pad_side),
                "y2":    min(img.shape[0], cy1 + y2 + pad_bot),
                "col":   col_label,
            })
            box_id += 1
    return img, boxes


# Run auto-detect
img, initial_boxes = auto_detect_boxes(SAMPLE_PAGE)
print(f"Auto-detected {len(initial_boxes)} boxes on {SAMPLE_PAGE.name}")

# Save initial boxes as JSON — the editor loads this as its starting state
initial_json_path = OUT / "initial_boxes.json"
initial_json_path.write_text(json.dumps({
    "page_image":  str(SAMPLE_PAGE.resolve()),
    "page_width":  img.shape[1],
    "page_height": img.shape[0],
    "boxes":       initial_boxes,
}, indent=2))
print(f"Initial boxes saved: {initial_json_path}")

Auto-detected 75 boxes on 1900-1901 (page 200).png
Initial boxes saved: output_box_fixing\initial_boxes.json


## 2. Build the interactive box editor

Opens as a single HTML file in the browser. Full page image on the left. Controls on the right.

### What you can do in the editor

- **Move a box** — click inside it and drag
- **Resize a box** — grab any corner or edge handle
- **Delete a box** — click the box, then press Delete (or the Delete button)
- **Add a new box** — hold Shift and drag on empty space
- **Save** — click "Save boxes" to download the corrected JSON

The saved JSON contains only the corrected box coordinates in full-page pixel space — no text, no fields, just geometry.

In [3]:
def build_box_editor_html(initial_json_path: Path, output_html_path: Path):
    """Generate the interactive box editor as a single self-contained HTML file."""
    data = json.loads(initial_json_path.read_text())
    page_uri = Path(data["page_image"]).resolve().as_uri()

    html = r"""<!DOCTYPE html>
<html><head><meta charset='utf-8'>
<title>Box editor</title>
<style>
  * { box-sizing: border-box; }
  body { font-family: Calibri, Arial, sans-serif; margin: 0; background: #f4f4f4; }
  .layout { display: flex; height: 100vh; overflow: hidden; }
  .canvas-panel { flex: 1; padding: 12px; overflow: auto; background: #333; }
  .canvas-wrap { position: relative; display: inline-block; }
  canvas { display: block; cursor: crosshair; }
  .side-panel { flex: 0 0 300px; padding: 16px; background: white;
                overflow: auto; border-left: 1px solid #ddd; }
  h2 { color: #1D6E5E; font-size: 18px; margin: 0 0 12px 0; }
  .stat { background: #E8F4F1; padding: 10px; border-radius: 4px; margin-bottom: 14px; }
  .stat b { color: #1D6E5E; }
  .instructions { font-size: 13px; color: #374151; line-height: 1.5; }
  .instructions kbd { background: #eee; border: 1px solid #ccc; padding: 1px 6px;
                      border-radius: 3px; font-family: monospace; font-size: 11px; }
  button { background: #1D6E5E; color: white; border: none; padding: 8px 16px;
           font-size: 14px; border-radius: 5px; cursor: pointer; margin: 4px 4px 4px 0;
           font-family: Calibri; }
  button.secondary { background: #6B7280; }
  button.danger { background: #DC2626; }
  button:hover { opacity: 0.9; }
  .zoom { margin: 10px 0; }
  .zoom input { width: 180px; }
  .selected-info { background: #fef3c7; padding: 8px; border-radius: 4px;
                   margin: 10px 0; font-size: 12px; }
</style></head><body>

<div class='layout'>
  <div class='canvas-panel'>
    <div class='canvas-wrap'>
      <canvas id='cv'></canvas>
    </div>
  </div>

  <div class='side-panel'>
    <h2>Box editor</h2>

    <div class='stat'>
      Total boxes: <b id='count'>0</b><br>
      Selected: <b id='selected-id'>none</b>
    </div>

    <div class='zoom'>
      Zoom: <span id='zoom-val'>100%</span><br>
      <input type='range' min='30' max='200' value='60' id='zoom-slider'
             oninput='setZoom(this.value)'>
    </div>

    <div>
      <button onclick='saveBoxes()'>Save boxes</button>
      <button class='secondary' onclick='resetToAuto()'>Reset to auto</button>
    </div>

    <div>
      <button class='danger' onclick='deleteSelected()'>Delete selected</button>
    </div>

    <div id='selection-info' class='selected-info' style='display:none;'></div>

    <div class='instructions'>
      <p><b>Controls:</b></p>
      <ul style='padding-left: 20px;'>
        <li>Click a box to select it</li>
        <li>Drag inside a selected box to move it</li>
        <li>Drag a corner/edge handle to resize</li>
        <li>Hold <kbd>Shift</kbd> and drag on empty space to draw a new box</li>
        <li>Press <kbd>Delete</kbd> or click "Delete selected"</li>
        <li>Use zoom slider to see detail</li>
      </ul>
      <p><b>When done:</b> click <b>Save boxes</b>. The corrected JSON downloads to your Downloads folder. Move it to <code>output_box_fixing/</code>.</p>
    </div>
  </div>
</div>

<script>
const PAGE_URI = "__PAGE_URI__";
const PAGE_W = __PAGE_W__;
const PAGE_H = __PAGE_H__;
const INITIAL_BOXES = __INITIAL_BOXES__;

// Box state — array of {id, x1, y1, x2, y2}
let boxes = JSON.parse(JSON.stringify(INITIAL_BOXES));
let nextId = boxes.length ? Math.max(...boxes.map(b => b.id)) + 1 : 0;
let selectedId = null;
let zoom = 0.6;

// Interaction state
let mode = "idle";   // idle | moving | resizing | drawing
let dragStartX = 0, dragStartY = 0;
let resizeCorner = null;
let originalBox = null;

const canvas = document.getElementById('cv');
const ctx = canvas.getContext('2d');
const pageImg = new Image();
pageImg.onload = () => { redraw(); };
pageImg.src = PAGE_URI;

function updateCanvasSize() {
  canvas.width = PAGE_W * zoom;
  canvas.height = PAGE_H * zoom;
}

function setZoom(v) {
  zoom = v / 100;
  document.getElementById('zoom-val').textContent = v + '%';
  updateCanvasSize();
  redraw();
}
updateCanvasSize();

function redraw() {
  ctx.clearRect(0, 0, canvas.width, canvas.height);
  ctx.drawImage(pageImg, 0, 0, canvas.width, canvas.height);
  boxes.forEach(b => {
    const isSelected = b.id === selectedId;
    ctx.strokeStyle = isSelected ? '#EF4444' : '#10B981';
    ctx.lineWidth = isSelected ? 3 : 2;
    ctx.strokeRect(b.x1 * zoom, b.y1 * zoom,
                   (b.x2 - b.x1) * zoom, (b.y2 - b.y1) * zoom);
    // Label
    ctx.fillStyle = ctx.strokeStyle;
    ctx.font = '11px Arial';
    ctx.fillText(b.id, b.x1 * zoom + 3, b.y1 * zoom + 12);
    // Handles when selected
    if (isSelected) {
      drawHandles(b);
    }
  });
  document.getElementById('count').textContent = boxes.length;
  document.getElementById('selected-id').textContent =
    selectedId === null ? 'none' : selectedId;
  updateSelectionInfo();
}

function drawHandles(b) {
  const HS = 8;
  const corners = [
    ['nw', b.x1, b.y1], ['ne', b.x2, b.y1],
    ['sw', b.x1, b.y2], ['se', b.x2, b.y2],
    ['n',  (b.x1+b.x2)/2, b.y1], ['s', (b.x1+b.x2)/2, b.y2],
    ['w',  b.x1, (b.y1+b.y2)/2], ['e', b.x2, (b.y1+b.y2)/2],
  ];
  ctx.fillStyle = '#EF4444';
  corners.forEach(([_, x, y]) => {
    ctx.fillRect(x*zoom - HS/2, y*zoom - HS/2, HS, HS);
  });
}

function updateSelectionInfo() {
  const el = document.getElementById('selection-info');
  if (selectedId === null) { el.style.display = 'none'; return; }
  const b = boxes.find(bb => bb.id === selectedId);
  if (!b) { el.style.display = 'none'; return; }
  el.style.display = 'block';
  el.innerHTML = `<b>Box ${b.id}</b><br>
    x: ${Math.round(b.x1)} to ${Math.round(b.x2)} (w=${Math.round(b.x2-b.x1)})<br>
    y: ${Math.round(b.y1)} to ${Math.round(b.y2)} (h=${Math.round(b.y2-b.y1)})`;
}

// Convert canvas coords to page pixel coords
function canvasToPage(e) {
  const rect = canvas.getBoundingClientRect();
  return {
    x: (e.clientX - rect.left) / zoom,
    y: (e.clientY - rect.top) / zoom,
  };
}

function findBoxAt(x, y) {
  // Return topmost box containing (x, y). Iterate reverse so latest drawn wins.
  for (let i = boxes.length - 1; i >= 0; i--) {
    const b = boxes[i];
    if (x >= b.x1 && x <= b.x2 && y >= b.y1 && y <= b.y2) return b;
  }
  return null;
}

function findHandleAt(x, y) {
  if (selectedId === null) return null;
  const b = boxes.find(bb => bb.id === selectedId);
  if (!b) return null;
  const HS = 8 / zoom;
  const corners = {
    nw: [b.x1, b.y1], ne: [b.x2, b.y1],
    sw: [b.x1, b.y2], se: [b.x2, b.y2],
    n:  [(b.x1+b.x2)/2, b.y1], s: [(b.x1+b.x2)/2, b.y2],
    w:  [b.x1, (b.y1+b.y2)/2], e: [b.x2, (b.y1+b.y2)/2],
  };
  for (const [name, [cx, cy]] of Object.entries(corners)) {
    if (Math.abs(x - cx) < HS && Math.abs(y - cy) < HS) return name;
  }
  return null;
}

canvas.addEventListener('mousedown', e => {
  const p = canvasToPage(e);
  if (e.shiftKey) {
    // Draw new box
    mode = "drawing";
    dragStartX = p.x; dragStartY = p.y;
    return;
  }
  const handle = findHandleAt(p.x, p.y);
  if (handle) {
    mode = "resizing";
    resizeCorner = handle;
    originalBox = {...boxes.find(b => b.id === selectedId)};
    dragStartX = p.x; dragStartY = p.y;
    return;
  }
  const box = findBoxAt(p.x, p.y);
  if (box) {
    selectedId = box.id;
    mode = "moving";
    dragStartX = p.x; dragStartY = p.y;
    originalBox = {...box};
  } else {
    selectedId = null;
  }
  redraw();
});

canvas.addEventListener('mousemove', e => {
  const p = canvasToPage(e);
  if (mode === "moving" && originalBox) {
    const dx = p.x - dragStartX;
    const dy = p.y - dragStartY;
    const b = boxes.find(bb => bb.id === selectedId);
    b.x1 = originalBox.x1 + dx;
    b.y1 = originalBox.y1 + dy;
    b.x2 = originalBox.x2 + dx;
    b.y2 = originalBox.y2 + dy;
    redraw();
  } else if (mode === "resizing" && originalBox) {
    const b = boxes.find(bb => bb.id === selectedId);
    if (resizeCorner.includes('n')) b.y1 = p.y;
    if (resizeCorner.includes('s')) b.y2 = p.y;
    if (resizeCorner.includes('w')) b.x1 = p.x;
    if (resizeCorner.includes('e')) b.x2 = p.x;
    // Keep coords valid
    if (b.x2 < b.x1) [b.x1, b.x2] = [b.x2, b.x1];
    if (b.y2 < b.y1) [b.y1, b.y2] = [b.y2, b.y1];
    redraw();
  } else if (mode === "drawing") {
    redraw();
    ctx.strokeStyle = '#3B82F6';
    ctx.lineWidth = 2;
    ctx.setLineDash([4, 4]);
    ctx.strokeRect(dragStartX * zoom, dragStartY * zoom,
                   (p.x - dragStartX) * zoom, (p.y - dragStartY) * zoom);
    ctx.setLineDash([]);
  }
});

canvas.addEventListener('mouseup', e => {
  const p = canvasToPage(e);
  if (mode === "drawing") {
    let x1 = Math.min(dragStartX, p.x);
    let y1 = Math.min(dragStartY, p.y);
    let x2 = Math.max(dragStartX, p.x);
    let y2 = Math.max(dragStartY, p.y);
    if (x2 - x1 > 10 && y2 - y1 > 10) {
      const newBox = { id: nextId++, x1, y1, x2, y2, col: "?" };
      boxes.push(newBox);
      selectedId = newBox.id;
    }
  }
  mode = "idle";
  originalBox = null;
  redraw();
});

document.addEventListener('keydown', e => {
  if (e.key === 'Delete' || e.key === 'Backspace') {
    if (selectedId !== null && document.activeElement === document.body) {
      deleteSelected();
      e.preventDefault();
    }
  }
});

function deleteSelected() {
  if (selectedId === null) return;
  boxes = boxes.filter(b => b.id !== selectedId);
  selectedId = null;
  redraw();
}

function resetToAuto() {
  if (!confirm('Discard all your edits and reset to the auto-detected boxes?')) return;
  boxes = JSON.parse(JSON.stringify(INITIAL_BOXES));
  nextId = boxes.length ? Math.max(...boxes.map(b => b.id)) + 1 : 0;
  selectedId = null;
  redraw();
}

function saveBoxes() {
  // Re-number boxes top-to-bottom, left-to-right so they're in reading order
  const sorted = [...boxes].sort((a, b) => {
    // Determine column by x-center of box
    const aCol = (a.x1 + a.x2) / 2 < PAGE_W / 2 ? 0 : 1;
    const bCol = (b.x1 + b.x2) / 2 < PAGE_W / 2 ? 0 : 1;
    if (aCol !== bCol) return aCol - bCol;
    return a.y1 - b.y1;
  });
  sorted.forEach((b, i) => { b.id = i; });
  const payload = {
    page_image: PAGE_URI,
    page_width: PAGE_W,
    page_height: PAGE_H,
    boxes: sorted,
    box_count: sorted.length,
  };
  const blob = new Blob([JSON.stringify(payload, null, 2)],
                        {type: 'application/json'});
  const url = URL.createObjectURL(blob);
  const a = document.createElement('a');
  a.href = url;
  a.download = 'corrected_boxes.json';
  a.click();
  URL.revokeObjectURL(url);
  alert('Saved ' + sorted.length + ' boxes to corrected_boxes.json.\n' +
        'Move the file into output_box_fixing/ and then run the next notebook cell.');
}
</script>
</body></html>"""
    html = (html
            .replace("__PAGE_URI__", page_uri)
            .replace("__PAGE_W__", str(data["page_width"]))
            .replace("__PAGE_H__", str(data["page_height"]))
            .replace("__INITIAL_BOXES__", json.dumps(data["boxes"])))
    output_html_path.write_text(html, encoding="utf-8")
    return output_html_path


editor_html = build_box_editor_html(
    OUT / "initial_boxes.json",
    OUT / "box_editor.html",
)
print(f"Box editor written to:\n  {editor_html.resolve()}")
print()
print("Open it in your browser. Fix the boxes. Click Save. Move the downloaded")
print("corrected_boxes.json into output_box_fixing/, then run the next cell.")

Box editor written to:
  C:\Users\ABHIRAMI.K\Downloads\output_box_fixing\box_editor.html

Open it in your browser. Fix the boxes. Click Save. Move the downloaded
corrected_boxes.json into output_box_fixing/, then run the next cell.


## 3. Verify the corrected boxes

After you save `corrected_boxes.json` and move it into `output_box_fixing/`, this cell loads it and draws the corrected boxes on the original page image so you can visually confirm the result before moving to step 2.

In [4]:
corrected_path = OUT / "corrected_boxes.json"

if not corrected_path.exists():
    print(f"Not found yet: {corrected_path}")
    print("Save the JSON from the editor and move it here first.")
else:
    data = json.loads(corrected_path.read_text())
    corrected_boxes = data["boxes"]
    print(f"Loaded {len(corrected_boxes)} corrected boxes")

    # Draw them on the page
    overlay = img.copy()
    for b in corrected_boxes:
        x1, y1, x2, y2 = int(b["x1"]), int(b["y1"]), int(b["x2"]), int(b["y2"])
        cv2.rectangle(overlay, (x1, y1), (x2, y2), (0, 200, 0), 2)
        cv2.putText(overlay, str(b["id"]), (x1 + 3, y1 + 18),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 200, 0), 1, cv2.LINE_AA)

    verify_path = OUT / "corrected_overlay.png"
    cv2.imwrite(str(verify_path), overlay)
    print(f"Verification overlay saved: {verify_path}")
    print()
    print("Open this PNG. Every entry on the page should have exactly ONE box.")
    print("If it does, step 1 is complete — you can move on to typing in the")
    print("field values (step 2) in the next notebook.")

Loaded 79 corrected boxes
Verification overlay saved: output_box_fixing\corrected_overlay.png

Open this PNG. Every entry on the page should have exactly ONE box.
If it does, step 1 is complete — you can move on to typing in the
field values (step 2) in the next notebook.
